# 26. 아미노산 변이 패턴으로 암종 구분하기 (H26, 개인 실험)

## 배경
- 22b: 미스센스의 **"무엇이 무엇으로"** 바뀌었는지(R→H 등) 비율 → CV 0.4487 (기준 0.4125 대비 +0.036)
- 이번 실험: **"어떤 아미노산이 어떤 방식으로"** 망가졌는지를 셈
  - 원래 아미노산 20종류 × 변이 유형 4종류(침묵·미스센스·넌센스·프레임시프트) = **80개**
  - in-frame indel은 22와 같은 이유로 제외 (train 3개 / test 3,501개 → 비율 분모에 넣으면 test에서만 나머지 칸 비율이 낮아짐)
  - 예: `K600fs` → K × 프레임시프트, `R213*` → R × 넌센스, `R132H` → R × 미스센스
  - 바뀐 아미노산이 아니라 **원래 아미노산** 기준: 프레임시프트는 바뀐 아미노산이 없어서 4가지 유형을 같은 기준으로 보려면 원래 아미노산이어야 함
- 기대하는 신호
  - DNA 복구 고장 암(MSI: COAD, STAD, UCEC)은 같은 염기가 반복되는 자리에서 프레임시프트가 잘 생김 (AAA → 라이신 K, TTT → 페닐알라닌 F, CCC → 프롤린 P) → "K × 프레임시프트" 같은 열
  - 22b와 달리 침묵·넌센스·프레임시프트까지 포함

## 근거 자료 (아이디어는 이전에 본 자료에서 착안, 아래 논문들이 이 방향을 뒷받침)
- **아미노산 단위 변이 패턴은 암종마다 다르다**: Szpiech ZA 외, *Prominent features of the amino acid mutation landscape in cancer*, PLoS ONE 12(8):e0183273 (2017)
  - 29개 암종의 아미노산 치환을 NMF·PCA로 분석 → 아미노산 변이 시그니처 6개 (COSMIC 독립 데이터에서 5개 재현)
  - 어떤 암은 R→H, 어떤 암은 E→K가 주를 이룸
- **원래 아미노산에 따라 변이 빈도가 다르다**: *Arginines of the CGN codon family are Achilles' heels of cancer genes*, Scientific Reports (2024)
  - 대부분의 암종에서 가장 많이 바뀌는 원래 아미노산은 아르기닌(R) (R→H, R→Q, R→C 순)
  - 원인은 주로 CpG 자리의 C>T 변이 → 원래 아미노산은 암마다 다른 DNA 변이 과정을 간접적으로 반영
- **변이 패턴 피처만으로 암종 분류가 가능하다**: Jiao W 외, *A deep learning system accurately classifies primary and metastatic cancers using passenger mutation patterns*, Nature Communications 11:728 (2020)
  - 승객 변이 패턴으로 24개 암종 분류 정확도 약 91%, 드라이버 변이 정보를 더하면 오히려 감소
- 외부 데이터 금지 규칙에 따라 논문은 아이디어 근거로만 사용, 피처는 대회 데이터의 변이 토큰에서만 계산

## 가설
- H26a: 80개 **개수**를 더하면 CV가 오른다 (요청한 방식)
- H26b: 개수 대신 환자 전체 변이 수로 나눈 **비율**이 test 변이 수 증가(2배)에 덜 흔들려 더 안정적이다 (11/11 회의 방향)
- H26c: 22b(현재 최고 피처)에 80개 비율을 더하면 22b보다 오른다

## 검증 방법
- 모델: 10 최적 설정 XGB 고정, 데이터: `load_data()` (정규화)
- 비교 (같은 fold)
  - 기준 `any + stats` vs 26a `+ 80개 개수` vs 26b `+ 80개 비율`
  - 22b `any + ratio + 치환 비율 상위 100` vs 26c `22b + 80개 비율`
- 한 환자 안에서만 계산 → Data Leakage 아님, 80개 목록은 도메인 지식으로 고정 (fold 안에서 고를 필요 없음)
- 22b의 `ratio`(유형별 비율, in-frame indel 제외)는 22 PR과 같은 계산을 이 노트북 안에 넣음 → 22 PR 머지 여부와 상관없이 실행 가능
- 채택 기준: 비교 대상보다 5 fold 중 4개 이상 상승
- 실행 환경: macOS M1 (XGB 점수는 데스크탑과 ±0.004 차이 가능), 제출 파일은 Windows에서만 저장

#### 준비: 데이터, 기본 피처, 22b 피처, 10 최적 설정

In [1]:
import sys; sys.path.append('..')
import re, time, platform, numpy as np, pandas as pd, xgboost as xgb
from collections import Counter
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline
from sklearn.utils.class_weight import compute_sample_weight
from src.config import SEED, WT
from src.data import load_data, get_gene_cols
from src.features import mutation_features, mutation_type, aa_substitution, MUT_TYPES
from src.validation import seed_everything, encode_target, get_folds, run_cv, save_submission

print(f'{platform.system()} | CPU {platform.machine()} | xgboost {xgb.__version__}')
seed_everything()
train, test, submission = load_data()
genes = get_gene_cols(train)
y, le = encode_target(train)
folds = get_folds(y)
F_tr, F_te = mutation_features(train, genes), mutation_features(test, genes)

class BalancedXGB(xgb.XGBClassifier):
    def fit(self, X, y, **kw):
        return super().fit(X, y, sample_weight=compute_sample_weight('balanced', y), **kw)

BEST = dict(max_depth=3, learning_rate=0.1, n_estimators=200, colsample_bytree=0.3,
            tree_method='hist', eval_metric='mlogloss', random_state=SEED, n_jobs=-1)

# 22a ratio (22 PR의 mutation_features()['ratio']와 같은 계산): 유형별 비율(in-frame indel 제외) + 전체 변이 수 log
def ratio_feats(df):
    keep = [t for t in MUT_TYPES if t != 'inframe_indel']
    rows = []
    for row in df[genes].to_numpy():
        c = Counter(mutation_type(t) for v in row if v != WT for t in v.split())
        tot = sum(c[t] for t in keep)
        r = {f'r_{t}': c[t] / max(tot, 1) for t in keep}
        r['n_mut_log'] = np.log1p(tot)
        rows.append(r)
    return pd.DataFrame(rows, index=df.index).astype(np.float32)

R_tr, R_te = ratio_feats(train), ratio_feats(test)

# 22b 피처 (22 노트북과 동일): 미스센스 치환 종류 비율, 상위 100개는 fold 학습 부분에서만
def sub_ratio_matrix(df):
    rows = []
    for row in df[genes].to_numpy():
        subs = (aa_substitution(t) for v in row if v != WT for t in v.split() if mutation_type(t) == 'missense')
        c = Counter(s for s in subs if s is not None)
        tot = sum(c.values())
        rows.append({k: n / tot for k, n in c.items()} if tot else {})
    M = pd.DataFrame(rows, index=df.index).fillna(0).astype(np.float32)
    M.columns = ['sub__' + k.replace('>', '_') for k in M.columns]
    return M

class TopNSubs(BaseEstimator, TransformerMixin):
    def __init__(self, n=100):
        self.n = n
    def fit(self, X, y=None):
        sub = [c for c in X.columns if c.startswith('sub__')]
        keep = set(X[sub].sum().sort_values(ascending=False).index[:self.n])
        self.drop_ = [c for c in sub if c not in keep]
        return self
    def transform(self, X):
        return X.drop(columns=self.drop_)

S_tr = sub_ratio_matrix(train)
S_te = sub_ratio_matrix(test).reindex(columns=S_tr.columns, fill_value=0)

Darwin | CPU arm64 | xgboost 3.4.1


#### 원래 아미노산 × 변이 유형 행렬 (80개)
- `ref_aa()`: 토큰 맨 앞의 원래 아미노산 (예: `R132H` → R, `K600fs` → K, `A630_P649del` → A, `1529_1530AL>AL` → A)
  - in-frame indel은 22와 같은 이유로 제외 (비율 분모에도 넣지 않음)
  - 위치만 있는 토큰(`277del`)이나 종결코돈(`*261*`)은 20종류에 없어서 제외
- 값: 환자별 해당 칸의 변이 **개수** (26a) / 개수를 환자의 전체 변이 수(80칸 합)로 나눈 **비율** (26b, 26c)

In [2]:
AA20 = 'ACDEFGHIKLMNPQRSTVWY'
TYPES4 = ['silent', 'missense', 'nonsense', 'frameshift']   # in-frame indel 제외 (22와 같은 이유)
AT_COLS = [f'at__{a}_{t}' for a in AA20 for t in TYPES4]
_LEAD = re.compile(r'^([A-Z*]+)\d')          # R132H, K600fs, WQ288fs, P233del, A630_P649del
_BLOCK = re.compile(r'^\d+_\d+([A-Z*]+)>')   # 1529_1530AL>AL

def ref_aa(token):
    m = _LEAD.match(token) or _BLOCK.match(token)
    r = m.group(1)[0] if m else None
    return r if r is not None and r in AA20 else None

def aa_type_counts(df):
    # 환자별 (원래 아미노산, 변이 유형) 개수 (한 환자 안에서만 계산 → 누수 아님)
    rows = []
    for row in df[genes].to_numpy():
        c = Counter()
        for v in row:
            if v == WT:
                continue
            for t in v.split():
                a, ty = ref_aa(t), mutation_type(t)
                if a is not None and ty in TYPES4:
                    c[f'at__{a}_{ty}'] += 1
        rows.append(c)
    return pd.DataFrame(rows, index=df.index).reindex(columns=AT_COLS, fill_value=0).fillna(0).astype(np.float32)

A_cnt_tr, A_cnt_te = aa_type_counts(train), aa_type_counts(test)
to_ratio = lambda A: A.div(A.sum(axis=1).clip(lower=1), axis=0).astype(np.float32)
A_rat_tr, A_rat_te = to_ratio(A_cnt_tr), to_ratio(A_cnt_te)
print('열 수:', A_cnt_tr.shape[1], '| train에서 한 번이라도 나온 칸:', int((A_cnt_tr.sum() > 0).sum()))
top = A_cnt_tr.sum().sort_values(ascending=False).head(8)
print('train에서 가장 많은 칸:', {k[4:]: int(v) for k, v in top.items()})

열 수: 80 | train에서 한 번이라도 나온 칸: 75
train에서 가장 많은 칸: {'R_missense': 26000, 'E_missense': 15541, 'S_missense': 13493, 'A_missense': 12649, 'P_missense': 12372, 'G_missense': 11771, 'D_missense': 10437, 'L_silent': 9156}


#### 개수 vs 비율: train vs test 평균 (관찰용)
- 개수는 test 변이가 2배라 커지고, 비율은 덜 변하는지 확인 (test 값은 피처 선택에 쓰지 않음)
- 프레임시프트 칸 예시 (MSI 관련)

In [3]:
fs_cols = [f'at__{a}_frameshift' for a in 'KFPEG']
pd.DataFrame({
    '개수 train': A_cnt_tr[fs_cols].mean(), '개수 test': A_cnt_te[fs_cols].mean(),
    '비율 train': A_rat_tr[fs_cols].mean(), '비율 test': A_rat_te[fs_cols].mean(),
}).round(4).rename(index=lambda s: s[4:]).assign(**{
    '전체 80칸 평균 (개수)': [f'train {A_cnt_tr.values.mean():.3f} / test {A_cnt_te.values.mean():.3f}'] + [''] * 4})

,개수 train,개수 test,비율 train,비율 test,전체 80칸 평균 (개수)
K_frameshift,0.1701,0.7455,0.0049,0.0052,train 0.498 / test 0.984
F_frameshift,0.0856,0.2549,0.0028,0.0024,
P_frameshift,0.1888,0.5247,0.0059,0.0050,
E_frameshift,0.0942,0.3189,0.0041,0.0044,
G_frameshift,0.1303,0.3928,0.0036,0.0037,


#### 비교: 기준 / 26a / 26b, 22b / 26c
- 시간이 오래 걸림 (XGB 5개, 약 17분)

In [4]:
cat = lambda parts: pd.concat(parts, axis=1)
SETS = {
    '기준: any + stats':            (cat([F_tr['any'], F_tr['stats']]), cat([F_te['any'], F_te['stats']]), None),
    '26a: 기준 + 80개 개수':        (cat([F_tr['any'], F_tr['stats'], A_cnt_tr]), cat([F_te['any'], F_te['stats'], A_cnt_te]), None),
    '26b: 기준 + 80개 비율':        (cat([F_tr['any'], F_tr['stats'], A_rat_tr]), cat([F_te['any'], F_te['stats'], A_rat_te]), None),
    '22b: any + ratio + 치환 100':   (cat([F_tr['any'], R_tr, S_tr]), cat([F_te['any'], R_te, S_te]), 100),
    '26c: 22b + 80개 비율':         (cat([F_tr['any'], R_tr, S_tr, A_rat_tr]), cat([F_te['any'], R_te, S_te, A_rat_te]), 100),
}
R = {}
for name, (X, X_te, n) in SETS.items():
    t = time.time()
    mk = (lambda: BalancedXGB(**BEST)) if n is None else (lambda: make_pipeline(TopNSubs(n), BalancedXGB(**BEST)))
    R[name] = run_cv(mk, X, y, X_te, folds=folds, verbose=False)
    print(f"{name}: CV {R[name]['cv_mean']:.4f} ± {R[name]['cv_std']:.4f} (OOF {R[name]['oof_score']:.4f}, {time.time() - t:.0f}s)")

fs = lambda k: np.array(R[k]['fold_scores'])
PAIR = {'26a: 기준 + 80개 개수': '기준: any + stats', '26b: 기준 + 80개 비율': '기준: any + stats',
        '26c: 22b + 80개 비율': '22b: any + ratio + 치환 100', '22b: any + ratio + 치환 100': '기준: any + stats'}
rows = []
for name in R:
    ref = PAIR.get(name)
    rows.append({'피처': name, 'CV': round(fs(name).mean(), 4), 'std': round(fs(name).std(), 4), 'OOF': round(R[name]['oof_score'], 4),
                 '비교 대상': ref or '-', '비교 대상 대비': round(fs(name).mean() - fs(ref).mean(), 4) if ref else '-',
                 '높은 fold': f'{int((fs(name) > fs(ref)).sum())}/5' if ref else '-'})
pd.DataFrame(rows)

기준: any + stats: CV 0.4125 ± 0.0109 (OOF 0.4131, 197s)
26a: 기준 + 80개 개수: CV 0.4387 ± 0.0058 (OOF 0.4398, 212s)
26b: 기준 + 80개 비율: CV 0.4424 ± 0.0096 (OOF 0.4430, 199s)
22b: any + ratio + 치환 100: CV 0.4487 ± 0.0065 (OOF 0.4497, 209s)
26c: 22b + 80개 비율: CV 0.4447 ± 0.0107 (OOF 0.4455, 217s)


,피처,CV,std,OOF,비교 대상,비교 대상 대비,높은 fold
0,기준: any + stats,0.4125,0.0109,0.4131,-,-,-
1,26a: 기준 + 80개 개수,0.4387,0.0058,0.4398,기준: any + stats,0.0261,5/5
2,26b: 기준 + 80개 비율,0.4424,0.0096,0.4430,기준: any + stats,0.0299,5/5
3,22b: any + ratio + 치환 100,0.4487,0.0065,0.4497,기준: any + stats,0.0362,5/5
4,26c: 22b + 80개 비율,0.4447,0.0107,0.4455,22b: any + ratio + 치환 100,-0.004,1/5


#### 중요도 상위 칸 (해석용)
- 26b를 train 전체로 한 번 학습해 어떤 칸을 쓰는지 확인 (점수 계산용 아님)

In [5]:
X26b = SETS['26b: 기준 + 80개 비율'][0]
m = BalancedXGB(**BEST).fit(X26b, y)
imp = pd.Series(m.feature_importances_, index=X26b.columns)
top = imp[imp.index.str.startswith('at__')].sort_values(ascending=False).head(12)
pd.DataFrame({'중요도': top.round(4)}).rename(index=lambda s: s[4:])

,중요도
W_frameshift,0.0032
P_missense,0.0014
E_missense,0.0014
G_nonsense,0.0014
F_silent,0.0014
R_missense,0.0013
L_silent,0.0012
Q_missense,0.0011
R_nonsense,0.0010
Q_nonsense,0.0010


#### test 예측 분포 (관찰용, 판단 기준 아님)

In [6]:
pd.DataFrame({k: pd.Series(le.classes_[v['test_proba'].argmax(1)]).value_counts(normalize=True)
              for k, v in R.items()}).fillna(0).sort_values('기준: any + stats', ascending=False).head(8).mul(100).round(1).astype(str) + '%'

,기준: any + stats,26a: 기준 + 80개 개수,26b: 기준 + 80개 비율,22b: any + ratio + 치환 100,26c: 22b + 80개 비율
STES,13.6%,17.1%,17.9%,21.8%,21.5%
COAD,10.9%,13.4%,12.3%,12.4%,12.6%
TGCT,5.5%,3.3%,3.1%,2.2%,2.0%
LUSC,5.1%,4.3%,4.2%,1.8%,2.1%
OV,4.9%,4.9%,4.6%,4.3%,4.5%
BRCA,4.9%,4.7%,4.6%,5.3%,5.1%
BLCA,4.4%,1.7%,1.3%,1.2%,0.9%
UCEC,3.9%,3.8%,4.5%,3.8%,3.5%


#### 26c + LR 섞기 vs C (11/12 추가)
- 22 후보 C(22b XGB 0.7 + LR 0.3)가 **LB 0.3650**으로 기준 → 26c도 같은 방식(XGB 0.7 + LR 0.3)으로 섞어 C와 같은 fold로 짝 비교
- LR은 14·C와 같은 설정 (any, C=0.1, balanced), 비율 0.7 / 0.3도 같음 (다시 고르지 않음)
- 채택 기준: C보다 5 fold 중 4개 이상 상승
- test COAD·STES 예측 비율도 같이 관찰 (판단 기준 아님)

In [7]:
from sklearn.linear_model import LogisticRegression
from src.validation import macro_f1

r_lr = run_cv(lambda: LogisticRegression(C=0.1, max_iter=3000, class_weight='balanced', random_state=SEED),
              F_tr['any'].astype('float32'), y, F_te['any'].astype('float32'), folds=folds, verbose=False)
blend = lambda r: (0.7 * r['oof_proba'] + 0.3 * r_lr['oof_proba'], 0.7 * r['test_proba'] + 0.3 * r_lr['test_proba'])
BL = {'C: 22b XGB 0.7 + LR 0.3 (LB 0.3650)': blend(R['22b: any + ratio + 치환 100']),
      '26c XGB 0.7 + LR 0.3': blend(R['26c: 22b + 80개 비율'])}
fsb = lambda P: np.array([macro_f1(y[va], P[va].argmax(1)) for _, va in folds])
bC = fsb(BL['C: 22b XGB 0.7 + LR 0.3 (LB 0.3650)'][0])
coad, stes = list(le.classes_).index('COAD'), list(le.classes_).index('STES')
rows = []
for name, (po, pt) in BL.items():
    s, pred = fsb(po), pt.argmax(1)
    rows.append({'조합': name, 'CV': round(s.mean(), 4), 'std': round(s.std(), 4), 'OOF': round(macro_f1(y, po.argmax(1)), 4),
                 'C 대비': round(s.mean() - bC.mean(), 4), 'C보다 높은 fold': f'{int((s > bC).sum())}/5',
                 'test COAD 예측': f'{np.mean(pred == coad):.1%}', 'test STES 예측': f'{np.mean(pred == stes):.1%}'})
print('fold별:', {k: np.round(fsb(v[0]), 4).tolist() for k, v in BL.items()})
pd.DataFrame(rows)

fold별: {'C: 22b XGB 0.7 + LR 0.3 (LB 0.3650)': [0.4594, 0.4437, 0.4564, 0.456, 0.4628], '26c XGB 0.7 + LR 0.3': [0.458, 0.4423, 0.4541, 0.4545, 0.4706]}


,조합,CV,std,OOF,C 대비,C보다 높은 fold,test COAD 예측,test STES 예측
0,C: 22b XGB 0.7 + LR 0.3 (LB 0.3650),0.4557,0.0065,0.4565,0.0000,0/5,15.0%,20.1%
1,26c XGB 0.7 + LR 0.3,0.4559,0.0091,0.4569,0.0002,1/5,15.4%,19.3%


#### 시드 평균 5개 (팀 규칙: 실험 결과물 + 시드 평균 결과물)
- 17과 같은 방식: XGB만 시드 5개(42, 0, 1, 2, 3)로 학습해 확률 평균, fold(seed 42)와 LR·비율 0.7 / 0.3은 고정
- C와 26c + LR 모두 시드 평균판을 만들어 같은 fold로 비교 (시드 42판은 위 결과를 그대로 사용)
- 시간이 오래 걸림 (XGB 8번 추가, 약 25분)

In [8]:
SEEDS = [42, 0, 1, 2, 3]

def xgb_seed_avg(name, n_sub):
    X, X_te, _ = SETS[name]
    oofs, tests = [R[name]['oof_proba']], [R[name]['test_proba']]   # 시드 42 = 위에서 돌린 결과
    for s in SEEDS[1:]:
        t = time.time()
        params = {**BEST, 'random_state': s}
        r = run_cv(lambda: make_pipeline(TopNSubs(n_sub), BalancedXGB(**params)), X, y, X_te, folds=folds, verbose=False)
        oofs.append(r['oof_proba']); tests.append(r['test_proba'])
        print(f"{name} 시드 {s}: CV {r['cv_mean']:.4f} ({time.time() - t:.0f}s)")
    return {'oof_proba': np.mean(oofs, axis=0), 'test_proba': np.mean(tests, axis=0)}

S22B, S26C = xgb_seed_avg('22b: any + ratio + 치환 100', 100), xgb_seed_avg('26c: 22b + 80개 비율', 100)
BL_S = {'C 시드 평균': blend(S22B), '26c + LR 시드 평균': blend(S26C)}
bCs = fsb(BL_S['C 시드 평균'][0])
rows = []
for name, (po, pt) in {**BL, **BL_S}.items():
    s, pred = fsb(po), pt.argmax(1)
    rows.append({'조합': name, 'CV': round(s.mean(), 4), 'std': round(s.std(), 4), 'OOF': round(macro_f1(y, po.argmax(1)), 4),
                 'C(시드 42)보다 높은 fold': f'{int((s > bC).sum())}/5', 'C 시드 평균보다 높은 fold': f'{int((s > bCs).sum())}/5',
                 'test COAD 예측': f'{np.mean(pred == coad):.1%}', 'test STES 예측': f'{np.mean(pred == stes):.1%}'})
pd.DataFrame(rows)

22b: any + ratio + 치환 100 시드 0: CV 0.4507 (213s)
22b: any + ratio + 치환 100 시드 1: CV 0.4566 (207s)
22b: any + ratio + 치환 100 시드 2: CV 0.4534 (208s)
22b: any + ratio + 치환 100 시드 3: CV 0.4546 (200s)
26c: 22b + 80개 비율 시드 0: CV 0.4450 (211s)
26c: 22b + 80개 비율 시드 1: CV 0.4480 (222s)
26c: 22b + 80개 비율 시드 2: CV 0.4444 (209s)
26c: 22b + 80개 비율 시드 3: CV 0.4515 (207s)


,조합,CV,std,OOF,C(시드 42)보다 높은 fold,C 시드 평균보다 높은 fold,test COAD 예측,test STES 예측
0,C: 22b XGB 0.7 + LR 0.3 (LB 0.3650),0.4557,0.0065,0.4565,0/5,1/5,15.0%,20.1%
1,26c XGB 0.7 + LR 0.3,0.4559,0.0091,0.4569,1/5,2/5,15.4%,19.3%
2,C 시드 평균,0.4622,0.0082,0.4631,4/5,0/5,15.0%,20.0%
3,26c + LR 시드 평균,0.4609,0.0104,0.4618,4/5,2/5,14.7%,20.2%


In [9]:
# 결과물 2개 (시드 42판 + 시드 평균판), 데스크탑(Windows)에서만 저장 · 채택 여부는 C와의 fold 비교로 판단
OUTS = {'blend_aatype26c_0.7_LR0.3': (BL['26c XGB 0.7 + LR 0.3'], bC),
        'blend_aatype26c_0.7_LR0.3_seed5': (BL_S['26c + LR 시드 평균'], bCs)}
for fname, ((po, pt), ref) in OUTS.items():
    up = int((fsb(po) > ref).sum())
    print(f"{fname}: 비교 대상(C)보다 높은 fold {up}/5 → {'채택 기준 통과' if up >= 4 else '기준 미달'}")
    if platform.system() == 'Windows':
        save_submission(submission, pt, le, name=fname, cv_score=macro_f1(y, po.argmax(1)))   # 파일명 숫자 = OOF
if platform.system() != 'Windows':
    print(f'{platform.system()}에서 실행 → 제출 파일 저장 안 함 (데스크탑에서 실행하면 저장)')

blend_aatype26c_0.7_LR0.3: 비교 대상(C)보다 높은 fold 1/5 → 기준 미달
blend_aatype26c_0.7_LR0.3_seed5: 비교 대상(C)보다 높은 fold 2/5 → 기준 미달
Darwin에서 실행 → 제출 파일 저장 안 함 (데스크탑에서 실행하면 저장)


#### 제출 파일 (데스크탑에서만 저장)
- 26 설정 중 비교 대상보다 5 fold 중 4개 이상 오른 것 가운데 CV 최고를 저장

In [10]:
cands = {k: R[k] for k in R if k.startswith('26') and (fs(k) > fs(PAIR[k])).sum() >= 4}
if not cands:
    print('채택 기준(비교 대상보다 5 fold 중 4개 이상)을 넘는 26 설정 없음 → 저장 안 함')
else:
    best = max(cands, key=lambda k: cands[k]['cv_mean'])
    print('저장 대상:', best)
    if platform.system() == 'Windows':
        save_submission(submission, cands[best]['test_proba'], le, name=f'xgb_d3_cs0.3_aatype26{best[2]}', cv_score=cands[best]['cv_mean'])
    else:
        print(f'{platform.system()}에서 실행 → 제출 파일 저장 안 함 (데스크탑에서 실행하면 저장)')

저장 대상: 26b: 기준 + 80개 비율
Darwin에서 실행 → 제출 파일 저장 안 함 (데스크탑에서 실행하면 저장)


## 결과 (macOS M1, in-frame indel 제외 80칸)

**미리 확인 (관찰용)**: 80칸 중 75칸이 train에 나타남. **개수**는 test에서 평균 약 2배(0.498 → 0.984, 예: K × 프레임시프트 0.17 → 0.75), **비율**은 거의 그대로(0.0049 → 0.0052)

| 피처 | CV (± std) | OOF | 비교 대상 | 대비 | 높은 fold |
| :--- | :-: | :-: | :--- | :-: | :-: |
| 기준: any + stats | 0.4125 ± 0.0109 | 0.4131 | - | - | - |
| 26a: 기준 + 80개 개수 | 0.4387 ± 0.0058 | 0.4398 | 기준 | +0.0261 | 5/5 |
| **26b: 기준 + 80개 비율** | **0.4424 ± 0.0096** | **0.4430** | 기준 | **+0.0299** | **5/5** |
| 22b: any + ratio + 치환 100 | 0.4487 ± 0.0065 | 0.4497 | 기준 | +0.0362 | 5/5 |
| 26c: 22b + 80개 비율 | 0.4447 ± 0.0107 | 0.4455 | 22b | −0.0040 | 1/5 |

- **26a·26b는 기준 대비 5/5 fold 상승** (+0.026 / +0.030) → 원래 아미노산 × 변이 유형 피처 자체는 효과 있음
- indel을 빼니 **비율(26b)이 개수(26a)보다 높아짐** → test 분포도 안정적인 비율 쪽이 CV에서도 나음
- **26c는 22b보다 낮아짐** (−0.004, 1/5 fold) → 22b에 더해도 추가 효과 없음
- 중요도 상위 칸: W × 프레임시프트, P·E·R × 미스센스, G × 넌센스, F·L × 침묵

**참고: indel 포함 100칸 버전(이전 실행)과 비교**

| 피처 | 100칸 (indel 포함) | 80칸 (indel 제외) |
| :--- | :-: | :-: |
| 26a | 0.4434 | 0.4387 |
| 26b | 0.4385 | 0.4424 |
| 26c | 0.4560 (22b 대비 +0.0073, 4/5) | 0.4447 (22b 대비 −0.0040, 1/5) |

- train에는 indel이 3개뿐이라 train 피처 값은 거의 같음 → 점수 차이는 대부분 **빈 indel 열 20개가 빠지면서 colsample(0.3)이 뽑는 열이 달라진 영향**
- 즉 이 정도(±0.005~0.01)는 열 구성만 바뀌어도 생기는 흔들림 → 이전 26c의 +0.0073(std 안)도 이 흔들림 범위였던 것으로 보임
- 기준·22b는 두 실행에서 같은 점수 (재현 확인)

**test 예측 분포 (관찰용)**
- STES: 기준 13.6% → 26a 17.1% · 26b 17.9% → 22b 21.8% · 26c 21.5%
- COAD: 10.9~13.4%로 큰 변화 없음, TGCT·LUSC·BLCA는 감소

**26c + LR 섞기 vs C, 시드 평균 5개 (11/12 추가, 같은 fold 짝 비교)**

| 조합 | CV (± std) | OOF | C(시드 42)보다 높은 fold | C 시드 평균보다 높은 fold | test COAD | test STES |
| :--- | :-: | :-: | :-: | :-: | :-: | :-: |
| C: 22b XGB 0.7 + LR 0.3 (LB 0.3650) | 0.4557 ± 0.0065 | 0.4565 | — | 1/5 | 15.0% | 20.1% |
| 26c XGB 0.7 + LR 0.3 | 0.4559 ± 0.0091 | 0.4569 | 1/5 (+0.0002) | 2/5 | 15.4% | 19.3% |
| C 시드 평균 (XGB 시드 5개) | 0.4622 ± 0.0082 | 0.4631 | 4/5 | — | 15.0% | 20.0% |
| 26c + LR 시드 평균 | 0.4609 ± 0.0104 | 0.4618 | 4/5 | 2/5 (−0.0013) | 14.7% | 20.2% |

- 26c + LR은 C와 평균이 거의 같지만 C보다 높은 fold는 1/5 (fold 4만 +0.0078, 나머지 4개 fold는 −0.001~−0.002) → **채택 기준 미달**
- 시드 평균끼리도 26c + LR 시드 평균이 C 시드 평균보다 2/5 fold, −0.0013 → 기준 미달
- test 예측 분포는 C와 거의 같음 (COAD 14.7~15.4%, STES 19.3~20.2%) → 80칸 비율이 22b + LR에 새 정보를 더하지 못함
- 시드별 CV: 22b 0.4487~0.4566, 26c 0.4444~0.4515 (시드 평균은 17과 같은 방식, M1 실행)

## 결론
- **H26a·H26b 채택 (CV 기준)**: 원래 아미노산 × 변이 유형 80칸이 기준 대비 5/5 fold 상승, **비율(26b)이 더 좋음** (+0.030)
- **H26c 기각**: 22b에 80칸 비율을 더해도 22b보다 낮음 (1/5 fold) → 22b가 이미 비슷한 정보를 담고 있음
- 이전(100칸) 26c의 상승은 열 구성에 따른 흔들림 범위였음 → 작은 CV 차이(std 안)는 채택 근거로 약함
- 제출 후보로는 22b 계열(22 후보 C)이 여전히 우선, 26b는 22b보다 CV가 낮아 단독 제출 우선순위는 낮음
- **26c + LR (시드 42·시드 평균) 기각**: C보다 1/5·2/5 fold → 26 계열은 제출 후보에서 제외, C 계열 유지
- 제출 파일: 데스크탑에서 실행하면 `xgb_d3_cs0.3_aatype26b` 저장 (채택 기준 통과 설정 중 CV 최고)
  - 섞기 결과물(팀 규칙): `blend_aatype26c_0.7_LR0.3`, `blend_aatype26c_0.7_LR0.3_seed5` (기준 미달, 기록용)
